# Log-linear classifier with NumPy

# Goal

Self-contained implementation of a log-linear classifier using NumPy from scratch. Every step spelled out for maximum readability.

Features:
1. **Forward pass**: Compute class probabilities using softmax, with multiclass support.
2. **Loss function**: Use cross-entropy loss for training.
3. **Backpropagation**: Implement gradient computation for weights (include biases by default) with vectorization.
4. **Update**: Built-in optimizer: stochastic gradient descent (SGD) with batch updates

In [1]:
import numpy as np

# Load Amazon sentiment dataset
import lxmls.readers.sentiment_reader as srs
from lxmls.deep_learning.utils import AmazonData

corpus = srs.SentimentCorpus("books")
data = AmazonData(corpus=corpus)

print(f"Dataset loaded:")
print(f"Training samples: {data.datasets['train']['input'].shape[0]}")
print(f"Test samples: {data.datasets['test']['input'].shape[0]}")
print(f"Feature dimension: {data.datasets['train']['input'].shape[1]}")
print(f"Number of classes: {len(np.unique(data.datasets['train']['output']))}")

Dataset loaded:
Training samples: 1600
Test samples: 400
Feature dimension: 13989
Number of classes: 2


In [ ]:
class LogLinearClassifier:
    """
    Supports multiclass classification with softmax activation and cross-entropy loss.
    """
    
    def __init__(self, input_size, num_classes, learning_rate=0.01):
        """
        Initialize the classifier parameters.
        
        Args:
            input_size: Number of input features
            num_classes: Number of output classes
            learning_rate: Step size for gradient descent
        """
        self.input_size = input_size
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        
        # Initialize weights with small random values (Xavier initialization)
        # Shape: (num_classes, input_size) - one weight vector per class
        scale = np.sqrt(2.0 / (input_size + num_classes))
        self.weights = np.random.normal(0, scale, (num_classes, input_size))
        
        # Initialize biases to zero
        # Shape: (num_classes,) - one bias per class
        self.biases = np.zeros(num_classes)
    
    def _logsumexp(self, x, axis=None, keepdims=False):
        """
        Compute log(sum(exp(x))) with numerical stability.
        This prevents overflow in softmax computation.
        
        The naive computation log(sum(exp(x))) can cause numerical issues:
        - If x contains large values, exp(x) can overflow to infinity
        - If x contains very negative values, exp(x) underflows to 0
        
        Solution: Use the log-sum-exp trick with max subtraction:
        log(sum(exp(x))) = log(sum(exp(x - max(x)) * exp(max(x))))
                         = log(exp(max(x)) * sum(exp(x - max(x))))
                         = max(x) + log(sum(exp(x - max(x))))
        
        This ensures that the largest value in (x - max(x)) is 0, preventing overflow.
        """
        # Find the maximum value along the specified axis
        max_x = np.max(x, axis=axis, keepdims=True)
        
        # Apply the log-sum-exp trick: subtract max before exp, then add it back
        # exp(x - max_x) ensures the largest exponential is exp(0) = 1
        return max_x + np.log(np.sum(np.exp(x - max_x), axis=axis, keepdims=keepdims))
    
    def forward(self, X):
        """
        Forward pass: compute class probabilities using softmax.
        
        Args:
            X: Input features, shape (batch_size, input_size)
            
        Returns:
            probabilities: Class probabilities, shape (batch_size, num_classes)
        """
        # Linear transformation: z = X @ W^T + b
        # X shape: (batch_size, input_size)
        # weights shape: (num_classes, input_size)
        # Result shape: (batch_size, num_classes)
        linear_output = np.dot(X, self.weights.T) + self.biases
        
        # Apply softmax in log domain for numerical stability
        # softmax(z_i) = exp(z_i) / sum(exp(z_j))
        # Using log-sum-exp trick: softmax(z) = exp(z - logsumexp(z))
        log_probs = linear_output - self._logsumexp(linear_output, axis=1, keepdims=True)
        probabilities = np.exp(log_probs)
        
        return probabilities
    
    def compute_loss(self, X, y):
        """
        Compute cross-entropy loss.
        
        Args:
            X: Input features, shape (batch_size, input_size)
            y: True labels, shape (batch_size,)
            
        Returns:
            loss: Average cross-entropy loss (scalar)
        """
        batch_size = X.shape[0]
        
        # Get class probabilities from forward pass
        probabilities = self.forward(X)
        
        # Compute cross-entropy loss: -log(P(y_true | x)) for a batch of samples
        # We need to extract the probability of the true class for each sample
        
        # Step 1: Create row indices [0, 1, 2, ..., batch_size-1]
        row_indices = np.arange(batch_size)
        
        # Step 2: Use indexing to select true class probabilities
        # probabilities[row_indices, y] extracts probabilities[i, y[i]] for each i
        # This gives us P(y_true | x) for each sample in the batch.
        # Recall that for multiclass cross-entropy, for each sample: 
        # only the log-prob of the true class matters, all other terms multiply by 0 and vanish.
        true_class_probabilities = probabilities[row_indices, y]
        
        # Step 3: Compute log-likelihood with numerical stability
        # Add small epsilon (1e-15) to prevent log(0) which would give -inf
        log_likelihood = np.log(true_class_probabilities + 1e-15)
        
        # Step 4: Cross-entropy loss is the negative mean log-likelihood
        loss = -np.mean(log_likelihood)
        
        return loss
    
    def compute_gradients(self, X, y):
        """
        Compute gradients using backpropagation (vectorized implementation).
        
        Args:
            X: Input features, shape (batch_size, input_size)
            y: True labels, shape (batch_size,)
            
        Returns:
            grad_weights: Gradient w.r.t. weights, shape (num_classes, input_size)
            grad_biases: Gradient w.r.t. biases, shape (num_classes,)
        """
        batch_size = X.shape[0]
        
        # Step 1: Forward pass to get predicted probabilities
        probabilities = self.forward(X)
        
        # Step 2: Create one-hot encoding of true labels
        # Initialize matrix of zeros with shape (batch_size, num_classes)
        y_onehot = np.zeros((batch_size, self.num_classes))
        
        # Use same indexing technique: set y_onehot[i, y[i]] = 1 for each sample i
        # This creates one-hot vectors where only the true class position is 1
        row_indices = np.arange(batch_size)
        y_onehot[row_indices, y] = 1
        
        # Step 3: Compute prediction error (gradient of loss w.r.t. linear output)
        # For softmax + cross-entropy, this simplifies to: predicted_probs - true_probs
        # Shape: (batch_size, num_classes)
        error = probabilities - y_onehot
        
        # Step 4: Compute gradient w.r.t. weights using matrix multiplication
        # We need: grad_W = (1/batch_size) * error^T @ X
        # - error^T has shape (num_classes, batch_size)
        # - X has shape (batch_size, input_size)  
        # - Result has shape (num_classes, input_size) matching self.weights
        grad_weights = np.dot(error.T, X) / batch_size
        
        # Step 5: Compute gradient w.r.t. biases
        # Sum the error over all samples (average across batch dimension)
        # Shape: (num_classes,) matching self.biases
        grad_biases = np.mean(error, axis=0)
        
        return grad_weights, grad_biases
    
    def update_parameters(self, grad_weights, grad_biases):
        """
        Update parameters using gradient descent.
        
        Args:
            grad_weights: Gradient w.r.t. weights
            grad_biases: Gradient w.r.t. biases
        """
        # SGD update: θ_new = θ_old - learning_rate * gradient
        self.weights -= self.learning_rate * grad_weights
        self.biases -= self.learning_rate * grad_biases
    
    def predict(self, X):
        """
        Make predictions on input data.
        
        Args:
            X: Input features, shape (batch_size, input_size)
            
        Returns:
            predictions: Predicted class labels, shape (batch_size,)
        """
        probabilities = self.forward(X)
        predictions = np.argmax(probabilities, axis=1)
        return predictions
    
    def accuracy(self, X, y):
        """
        Compute classification accuracy.
        
        Args:
            X: Input features, shape (batch_size, input_size)
            y: True labels, shape (batch_size,)
            
        Returns:
            accuracy: Classification accuracy (float between 0 and 1)
        """
        predictions = self.predict(X)
        accuracy = np.mean(predictions == y)
        return accuracy

In [ ]:
def create_batches(X, y, batch_size):
    """
    Create mini-batches from dataset.
    
    Args:
        X: Input features, shape (num_samples, input_size)
        y: Labels, shape (num_samples,)
        batch_size: Size of each batch
        
    Yields:
        (X_batch, y_batch): Tuples of batched data
    """
    num_samples = X.shape[0]
    indices = np.arange(num_samples)
    np.random.shuffle(indices)  # Shuffle for each epoch
    
    for start_idx in range(0, num_samples, batch_size):
        end_idx = min(start_idx + batch_size, num_samples)
        batch_indices = indices[start_idx:end_idx]
        yield X[batch_indices], y[batch_indices]

def train_model(model, X_train, y_train, X_test, y_test, num_epochs, batch_size):
    """
    Training loop with SGD and progress monitoring.
    
    Args:
        model: LogLinearClassifier instance
        X_train, y_train: Training data
        X_test, y_test: Test data  
        num_epochs: Number of training epochs
        batch_size: Mini-batch size
    """
    print("Starting training...")
    print(f"Initial test accuracy: {model.accuracy(X_test, y_test):.4f}")
    print("-" * 50)
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        # Process all batches in one epoch
        for X_batch, y_batch in create_batches(X_train, y_train, batch_size):
            # Compute gradients for current batch
            grad_weights, grad_biases = model.compute_gradients(X_batch, y_batch)
            
            # Update parameters
            model.update_parameters(grad_weights, grad_biases)
            
            # Track loss for monitoring
            batch_loss = model.compute_loss(X_batch, y_batch)
            epoch_loss += batch_loss
            num_batches += 1
        
        # Compute average loss and accuracy for this epoch
        avg_loss = epoch_loss / num_batches
        train_accuracy = model.accuracy(X_train, y_train)
        test_accuracy = model.accuracy(X_test, y_test)
        
        # Print progress
        print(f"Epoch {epoch+1:2d}/{num_epochs}: "
              f"Loss = {avg_loss:.4f}, "
              f"Train Acc = {train_accuracy:.4f}, "
              f"Test Acc = {test_accuracy:.4f}")

In [4]:
# Prepare data
X_train = data.datasets['train']['input']
y_train = data.datasets['train']['output']
X_test = data.datasets['test']['input']
y_test = data.datasets['test']['output']

print("== Data shapes ==")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")
print(f"Classes: {np.unique(y_train)}")

# Instantiate the model
input_size = X_train.shape[1]
num_classes = len(np.unique(y_train))
learning_rate = 0.05

model = LogLinearClassifier(
    input_size=input_size,
    num_classes=num_classes,
    learning_rate=learning_rate
)

print(f"\n== Model initialized ==")
print(f"Learning rate: {learning_rate}")
print(f"Number of classes: {num_classes}")
print(f"Weight matrix shape: {model.weights.shape}")
print(f"Bias vector shape: {model.biases.shape}")

== Data shapes ==
X_train: (1600, 13989)
y_train: (1600,)
X_test: (400, 13989)
y_test: (400,)
Classes: [0 1]

== Model initialized ==
Learning rate: 0.05
Number of classes: 2
Weight matrix shape: (2, 13989)
Bias vector shape: (2,)


In [5]:
# Training hyperparameters
num_epochs = 10
batch_size = 30

# Reinitialize model to ensure fresh start (prevents continued training from previous runs)
model = LogLinearClassifier(
    input_size=input_size,
    num_classes=num_classes,
    learning_rate=learning_rate
)

# Train the model
train_model(
    model=model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    num_epochs=num_epochs,
    batch_size=batch_size
)

Starting training...
Initial test accuracy: 0.4375
--------------------------------------------------
Epoch  1/10: Loss = 0.4759, Train Acc = 0.8519, Test Acc = 0.7475
Epoch  2/10: Loss = 0.3672, Train Acc = 0.8925, Test Acc = 0.7650
Epoch  3/10: Loss = 0.3110, Train Acc = 0.9256, Test Acc = 0.8175
Epoch  4/10: Loss = 0.2740, Train Acc = 0.9487, Test Acc = 0.8225
Epoch  5/10: Loss = 0.2480, Train Acc = 0.9587, Test Acc = 0.8250
Epoch  6/10: Loss = 0.2256, Train Acc = 0.9650, Test Acc = 0.8275
Epoch  7/10: Loss = 0.2080, Train Acc = 0.9738, Test Acc = 0.8250
Epoch  8/10: Loss = 0.1947, Train Acc = 0.9788, Test Acc = 0.8325
Epoch  9/10: Loss = 0.1837, Train Acc = 0.9850, Test Acc = 0.8300
Epoch 10/10: Loss = 0.1719, Train Acc = 0.9862, Test Acc = 0.8275
